In [1]:
hello = "Hello, World!"

In [2]:
hello

'Hello, World!'

In [43]:
from sentence_transformers import SentenceTransformer   

In [44]:
Q1 = "I just discovered the course. Can I still join it?"
Q2 = "I just found out about the program. Can I still enroll?"

In [45]:
model = SentenceTransformer("all-MiniLM-L6-v2")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [6]:
q1 = "Can I still join the course after the start date?"
v1 = model.encode(q1)

In [7]:
v1

array([ 2.13904232e-02, -7.39800036e-02,  1.42067147e-03,  2.13816538e-02,
        2.45113056e-02,  3.15582789e-02, -1.10839739e-01, -1.05017491e-01,
       -6.18259013e-02, -6.42310968e-03,  3.72400018e-03,  9.06393453e-02,
       -9.49938875e-03,  6.53976798e-02,  1.10946437e-02, -2.10097469e-02,
       -3.35125439e-02, -4.31677662e-02,  9.96348634e-03,  1.41969854e-02,
       -6.40415028e-02, -7.04182126e-03, -7.91187733e-02,  5.80030866e-02,
        1.30209571e-03,  4.19734837e-03,  5.70979118e-02,  6.39447793e-02,
        2.49902792e-02, -3.95876840e-02, -3.79506461e-02,  2.70394497e-02,
        1.79423746e-02,  1.72272492e-02,  3.43311131e-02,  9.29055270e-03,
        5.86054958e-02, -4.97789681e-02, -5.05369995e-03,  4.34328392e-02,
       -1.56622808e-02, -2.97534633e-02, -5.13327774e-03,  5.13414890e-02,
        6.16063876e-03,  6.86980486e-02, -1.29505536e-02, -5.61938547e-02,
       -1.08265029e-02,  5.96684217e-02,  5.29939793e-02, -3.42755094e-02,
       -4.15274128e-02, -

In [8]:
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

In [9]:
v1.dot(dv)

np.float32(0.3233239)

In [10]:
q2 = "How to install Docker on Windows?"
v2 = model.encode(q2)

In [11]:
v2.dot(v1)

np.float32(-0.14271769)

In [12]:
import requests
# from minsearch import index
def load_faq_documents():
    docs_url = "https://datatalks.club/faq/json/courses.json"
    response = requests.get(docs_url)
    courses_raw = response.json()

    documents = []
    url_prefix = "https://datatalks.club/faq"

    for course in courses_raw:
        course_url = f"""{url_prefix}{course["path"]}"""

        course_response = requests.get(course_url)
        course_response.raise_for_status()
        course_data = course_response.json()

        documents.extend(course_data)

    return documents

In [13]:
documents = load_faq_documents()

In [14]:
documents[10]

{'id': '316180784f',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: How many hours per week am I expected to spend on this course?',
 'answer': 'It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week.\n\nYou can also calculate it yourself using [this data](https://github.com/DataTalksClub/zoomcamp-analytics/tree/main/data/de-zoomcamp-2023) and then update this answer.'}

In [15]:
text =[]
for doc in documents:
    text.append(doc['question'] + " " + doc['answer'])
    

In [16]:
from tqdm import tqdm
batch_size = 32
vectors = []
for i in tqdm(range(0, len(text), batch_size)):
    batch = text[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)
len(vectors)

100%|██████████| 44/44 [01:43<00:00,  2.36s/it]


1401

In [18]:
import numpy as np
X = np.array(vectors)

In [19]:
query = "Can I still join the course after the start date?"
v_query = model.encode(query)

In [20]:
scores = X.dot(v_query)

In [21]:
scores = [v_query.dot(X[i]) for i in range(len(X))]

In [22]:
idx = np.argmax(scores)
idx, scores[idx]

(np.int64(2), np.float32(0.762941))

In [23]:
top5 = np.argsort(scores)[-5:] # the -5 means the last 5 elements, which are the highest scores because argsort sorts in ascending order

In [ ]:
top5 # they came out smallest to largest, so we need to reverse it

array([   7, 1009,  567, 1150,    2])

In [ ]:
top5 = top5[::-1] # reverse the order to get the highest scores first
top5

array([   2, 1150,  567, 1009,    7])

In [26]:
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

0.762941
{'id': '3f1424af17', 'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

0.7579372
{'id': '2d8b16c2a0', 'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute."}

0.7192131
{'id': '41aabbd7c5', 'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related 

In [47]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, documents)

NameError: name 'X' is not defined

In [30]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vindex.search(query_vector, filter_dict={"course":"llm-zoomcamp"}, num_results=5)

In [31]:
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nTo get the certificate, you need to finish a capstone project and complete the\nrequired peer reviews. Homework is not required. You can work through the\nmaterial and prepare your project in self-paced mode, but project submission and\npeer review must happen while a live cohort is accepting them.'},
 {'id': 'bd31146b0e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  '

In [37]:
class RAGBase:

    def __init__(
        self,
        index,
        llm_client,
        instructions=INSTRUCTIONS,
        prompt_template=PROMPT_TEMPLATE,
        course='llm-zoomcamp',
        model='nvidia/nemotron-3-ultra-550b-a55b:free'  # <-- change default to a real OpenRouter free model
    ):
        self.index = index
        self.llm_client = llm_client
        self.instructions = instructions
        self.course = course
        self.prompt_template = prompt_template
        self.model = model

    def search(self, query, num_results=5):
        boost_dict = {'question': 3.0, 'section': 0.5}
        filter_dict = {'course': self.course}
        return self.index.search(
            query,
            num_results=num_results,
            boost_dict=boost_dict,
            filter_dict=filter_dict
        )

    def build_context(self, search_results):
        lines = []
        for doc in search_results:
            lines.append(doc['section'])
            lines.append('Q: ' + doc['question'])
            lines.append('A: ' + doc['answer'])
            lines.append('')
        return '\n'.join(lines).strip()

    def build_prompt(self, query, search_results):
        context = self.build_context(search_results)
        return self.prompt_template.format(
            question=query, context=context
        )

    def llm(self, prompt):
        input_messages = [
            {'role': 'system', 'content': self.instructions},   # 'developer' -> 'system'
            {'role': 'user', 'content': prompt}
        ]

        response = self.llm_client.chat.completions.create(     # responses -> chat.completions
            model=self.model,
            messages=input_messages,                             # input= -> messages=
            max_tokens=1000,
        )

        return response.choices[0].message.content              # output_text -> this

    def rag(self, query):
        search_results = self.search(query)
        prompt = self.build_prompt(query, search_results)
        answer = self.llm(prompt)
        return answer

In [ ]:
#override the class to suit the vector search
class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

In [ ]:
import requests
from minsearch import Index


def load_faq_data():
    #load our chunked data 
    return documents


def build_index(documents):
    index = Index(
        text_fields=['question', 'section', 'answer'],
        keyword_fields=['course']
    )
    index.fit(documents)
    return index

In [24]:
# ============================================================
# 1. IMPORTS + API KEY
# ============================================================

import os
import requests
from dotenv import load_dotenv
from minsearch import Index
from openai import OpenAI


# Load variables from .env
loaded = load_dotenv()

print("dotenv loaded:", loaded)
print("API key present:", bool(os.getenv("OPENROUTER_API_KEY")))


load_dotenv()

openai_client = OpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
)


dotenv loaded: True
API key present: True


In [25]:
documents = load_faq_data()
index = build_index(documents)


In [38]:
assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    model='nvidia/nemotron-3-ultra-550b-a55b:free',
)

In [39]:
query = "I just found out about the program, can I still sign up?"
assistant.rag(query)    

'Yes, you can still join. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted.'

In [46]:
vector_assistant = RAGVector(
    embedder=model,
    index=vindex,
    llm_client=openai_client,
)
vector_assistant.rag("the program has already begun, can I still sign up?")


NameError: name 'vindex' is not defined